# 14 — Freeze-Train Alignment Stage → new init artifact

Probe verdict (nb13): v5 embeddings carry strong crosslingual signal (median rank 16/30522) that
73-step CPT failed to use — and the scaled head is slightly miscalibrated at step 0
(full 7.36 > bias_only 7.27). This stage trains **embeddings + LM head ONLY** (28-layer body
frozen) so encoder/decoder align with the body before full CPT — the historical stage-1 recipe.

**Output:** `init/trung_salt_freezealigned/` — a normal init artifact for the bake-off.

| param | value | why |
|---|---|---|
| trainable | `model.encoder.weight`, `decoder.weight`, `decoder.bias` (~47M) | align entry/exit doors, keep body intact |
| data | 20,000 chunks from the existing 40k-doc cache (~20M tokens) | instant load, distinct from the 100k CPT data |
| effective batch | 128 (32 × GA 4) → **~156 optimizer steps** | "train a bit", not a CPT |
| LR / schedule | **1e-4**, cosine, warmup 10% | historical stage-1 value; high enough to move 47M params in 156 steps |
| weight decay | **0.0** | AdamW decay is gradient-independent — it would shrink rare-token rows and erode the verified geometry for nothing |
| MLM prob | 0.20 | match NeoBERT/CPT |
| checkpoints | local `/content`, `save_strategy='no'` | only the final tensors matter; artifact written at the end |

In [ ]:
%%capture
!pip install -U transformers accelerate datasets safetensors sentencepiece tokenizers pandas tqdm huggingface_hub

In [ ]:
import sys, json, math, shutil, glob, hashlib
from pathlib import Path
import torch
import torch.nn.functional as F

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)
PROJECT_ROOT = Path('/content/drive/MyDrive/SALT3')
sys.path.insert(0, str(PROJECT_ROOT / 'code')); sys.path.insert(0, '/content')
import importlib
import salt3_common as sc; importlib.reload(sc)
import salt3_decoder_variants as sdv; importlib.reload(sdv)
import salt3_init_signal_probe as probe; importlib.reload(probe)
from salt3_common import (configure_environment, set_seed, ensure_dir, load_model_safe,
                          make_mlm_datasets, make_mlm_collator, make_neo_mlm_trainer_class,
                          training_args, eval_with_perplexity, JsonlMetricsCallback)
configure_environment(); set_seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE == 'cuda', 'freeze-train needs the GPU runtime (use the A100 — L4 forward NaNs)'

SRC_INIT_NAME = 'trung_salt_globalmap_freqbias'
OUT_INIT_NAME = 'trung_salt_freezealigned'
SRC_INIT_DIR  = PROJECT_ROOT / 'init' / SRC_INIT_NAME
OUT_INIT_DIR  = ensure_dir(PROJECT_ROOT / 'init' / OUT_INIT_NAME)
DATASET_CACHE = PROJECT_ROOT / 'datasets'

FREEZE_CONFIG = {
    'num_examples': 40_000,      # reuse the existing tokenized cache (instant)
    'num_chunks': 20_000,        # ~20M tokens for the alignment stage
    'max_seq_len': 1024, 'eval_ratio': 0.02, 'mlm_probability': 0.20, 'seed': 42,
    'per_device_train_batch_size': 32, 'per_device_eval_batch_size': 32,
    'gradient_accumulation_steps': 4,    # effective batch 128 -> ~156 steps
    'learning_rate': 1e-4, 'warmup_ratio': 0.10, 'weight_decay': 0.0,
    'adam_beta2': 0.95, 'logging_steps': 10, 'eval_steps': 25,
}
print('source:', SRC_INIT_DIR, '\noutput:', OUT_INIT_DIR)

## 1. Load v5 and freeze the body

In [ ]:
model = load_model_safe(SRC_INIT_DIR / 'model', device=DEVICE)
tokenizer = __import__('transformers').AutoTokenizer.from_pretrained(SRC_INIT_DIR / 'model', trust_remote_code=True)

for p in model.parameters():
    p.requires_grad_(False)
for p in (model.model.encoder.weight, model.decoder.weight, model.decoder.bias):
    p.requires_grad_(True)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'trainable {trainable/1e6:.1f}M / {total/1e6:.1f}M params '
      f'({trainable/total:.1%}) — embeddings + decoder + bias only')

# fingerprints to verify the body really stays frozen and the heads really move
def sha(t): return hashlib.sha256(t.detach().cpu().float().numpy().tobytes()).hexdigest()[:12]
body_ref = model.model.transformer_encoder[0].qkv.weight
fp_body_before = sha(body_ref)
fp_emb_before = sha(model.model.encoder.weight)
print('body fp:', fp_body_before, '| emb fp:', fp_emb_before)

## 2. Data (cached) + collator

In [ ]:
cfg = FREEZE_CONFIG
train_dataset, eval_dataset = make_mlm_datasets(
    tokenizer=tokenizer, cache_dir=DATASET_CACHE,
    num_examples=cfg['num_examples'], max_seq_len=cfg['max_seq_len'],
    num_chunks=cfg['num_chunks'], eval_ratio=cfg['eval_ratio'], seed=cfg['seed'])
data_collator = make_mlm_collator(tokenizer, mlm_probability=cfg['mlm_probability'])
print('train chunks:', len(train_dataset), '| eval chunks:', len(eval_dataset))

## 3. Train the alignment stage

In [ ]:
NeoMLMTrainer = make_neo_mlm_trainer_class()
args = training_args(
    '/content/freeze_ckpt',
    num_train_epochs=1, max_steps=-1,
    per_device_train_batch_size=cfg['per_device_train_batch_size'],
    per_device_eval_batch_size=cfg['per_device_eval_batch_size'],
    gradient_accumulation_steps=cfg['gradient_accumulation_steps'],
    learning_rate=cfg['learning_rate'], warmup_ratio=cfg['warmup_ratio'],
    weight_decay=cfg['weight_decay'], adam_beta2=cfg['adam_beta2'],
    logging_steps=cfg['logging_steps'], eval_steps=cfg['eval_steps'],
    save_strategy='no', load_best_model_at_end=False, run_name='freeze-align-v5')
metrics = JsonlMetricsCallback(OUT_INIT_DIR / 'freeze_train_metrics.jsonl')
trainer = NeoMLMTrainer(model=model, args=args, train_dataset=train_dataset,
                        eval_dataset=eval_dataset, data_collator=data_collator,
                        processing_class=tokenizer, callbacks=[metrics.callback])

pre = eval_with_perplexity(trainer)
print('pre-align eval :', pre)
trainer.train()
post = eval_with_perplexity(trainer)
print('post-align eval:', post)
print(f"alignment gain : {pre['eval_loss'] - post['eval_loss']:+.4f} nats")

assert sha(body_ref) == fp_body_before, 'BODY CHANGED — freeze failed!'
assert sha(model.model.encoder.weight) != fp_emb_before, 'embeddings did not move'
print('✓ body frozen, heads trained')

## 4. Save as a new init artifact

Same mechanism as every other arm: copy v5's model dir (patched files, config, tokenizer,
body — the body is bit-identical since it was frozen) and swap the three trained tensors.

In [ ]:
sdv.write_artifact(SRC_INIT_DIR, OUT_INIT_DIR, {
    'model.encoder.weight': model.model.encoder.weight.detach().cpu(),
    'decoder.weight': model.decoder.weight.detach().cpu(),
    'decoder.bias': model.decoder.bias.detach().cpu(),
}, {
    'init_name': OUT_INIT_NAME,
    'stage1_freeze_align': {k: cfg[k] for k in
        ('num_chunks', 'learning_rate', 'warmup_ratio', 'weight_decay',
         'gradient_accumulation_steps', 'per_device_train_batch_size')},
    'stage1_pre_eval_loss': round(pre['eval_loss'], 4),
    'stage1_post_eval_loss': round(post['eval_loss'], 4),
})
# copy anchor CSVs so the probe excludes the SAME anchor set as for v5
for csv in SRC_INIT_DIR.glob('*.csv'):
    shutil.copy(csv, OUT_INIT_DIR / csv.name)
print('artifact ready: init/%s/model' % OUT_INIT_NAME)

## 5. Probe the aligned artifact — did alignment preserve the crosslingual structure?

v5 scored median rank 16 / acc@10 40.8% before alignment. If alignment training *erodes* that
(ranks ballooning), the LR is too hot — drop to 5e-5 and rerun. Mild drift is expected and fine.

In [ ]:
del trainer, model; torch.cuda.empty_cache()
res = probe.run_probe(OUT_INIT_DIR, device='cpu')
v5_probe_path = SRC_INIT_DIR / 'init_signal_probe.json'
if v5_probe_path.exists():
    v5p = json.loads(v5_probe_path.read_text())['salt']
    a = res['salt']
    print(f"\nretrieval drift v5 -> freezealigned: median {v5p['median_rank']} -> {a['median_rank']}, "
          f"acc@10 {v5p['acc10']:.1%} -> {a['acc10']:.1%}")

## 6. Init inventory for the 100k-doc CPT bake-off

| # | artifact | status | priority |
|---|---|---|---|
| 1 | `trung_salt_globalmap_freqbias` | ready, signal-verified | **must run** (main) |
| 2 | `trung_random_meannorm` | ready, A/B-verified | **must run** (control) |
| 3 | `trung_salt_freezealigned` | built by THIS notebook | **must run** (utilization fix) |
| 4 | `trung_salt_dectied` | build via 01c (minutes) | high — exposes verified geometry directly in logits |
| 5 | `trung_salt_decscale10` | build via 01c | high — is ×0.1 muting the signal? |
| 6 | `trung_salt_decscale05` | build via 01c | medium |
| 7 | `trung_salt_decpertoken` | build via 01c | low — historical loser, retest if budget |

Legacy (NEVER train): v1, v2, v2_fix, x1, x2_tied, v3_proj — artifact-side defects, forensic relics.

**nb02 settings per arm:** `MODE='new'`, `RUN_NAME='cpt100k-<arm>'`, `num_examples=100_000`,
`num_chunks=None`, same seed 42. ~93k chunks → ~182 optimizer steps at effective batch 512 →
set `eval_steps=20` (≈9 curve points) and `save_steps=60`. Identical settings across ALL arms;
compare curves at matched steps, then nb03 downstream for the winner. Run on **A100 only**.